In [21]:
import json
import uuid

num_items = 5000

# generate random key-value pairs
data = {str(uuid.uuid4()): str(uuid.uuid4()) for _ in range(num_items)}

# save to file
with open("./tmp/needles.json", "w") as f:
    json.dump(data, f, indent=2)

In [ ]:
import json
with open('./tmp/needles.json','r',encoding='utf-8') as f:
    data = json.load(f)
    
prompt_template = """
You are a needle-in-the-haystack expert extractor.
Your only task is extracting the value corresponding to a specified key in the JSON object below. 

JSON data: 
```json
{data}
```

Given a key, extract its corresponding value, without any additional text or explanation.
Hard rule: if you cannot find the key in the JSON data, respond with "NOT_FOUND".

<example>
user: 5a5c979b-0a4f-43ff-bd05-5ca447015a0d
assistant: 258e8933-0533-42c8-a0a5-1e0d056fbe9
</example>

user: {key}
assistant:
"""

In [11]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.chat_models import init_chat_model

def _test(model,batches = [10, 100, 250, 500, 1000]):
    llm = init_chat_model(
    model=model,
    temperature=0,
    )
    print(f"\n===== model: {model} =====")
    for num in batches:
        _data = {k: data[k] for k in list(data.keys())[:num]}
        token_count = len(json.dumps(_data))
        first_needle = dict(list(_data.items())[:1])
        middle_needle = dict(list(_data.items())[num // 2:num // 2 + 1])
        third_quarter_needle = dict(list(_data.items())[3 * num // 4:3 * num // 4 + 1])
        before_final_needle = dict(list(_data.items())[-2:-1])
        final_needle = dict(list(_data.items())[-1:])
        not_found_needle = {"296845b6-70e4-447d-adf0-47b1c9c0e9a8": "NOT_FOUND"}
        print(f"--- n: {num}  ---")
        for needle_dict in [not_found_needle,first_needle, middle_needle, third_quarter_needle, before_final_needle, final_needle]:
            needle = list(needle_dict.keys())[0]
            prompt = prompt_template.format(data=json.dumps(_data, indent=2), key=needle)
            messages = [
                SystemMessage(content=prompt)
            ]
            response = llm.invoke(messages)
            print(f"{'✅' if response.content.strip() == needle_dict[needle] else '❌'} {needle} : {needle_dict[needle]} -> {response.content.strip()}")

In [ ]:
model = "ollama:granite4:3b" #context size 128k 
_test(model)

model = "ollama:gpt-oss:20b" #context size 131k
_test(model)
#note: same error can be pure hallucination, other simply "wrong" (valid value but not the correct one, suggesting that the model has a vague memory of the data)


===== model: ollama:granite4:3b =====
--- n: 10  ---
✅ 196845b6-70e4-447d-adf0-47b1c9c0e9a8 : NOT_FOUND -> NOT_FOUND
✅ 0876ba2c-ee0e-4e05-bc7e-7464ef3f1df9 : 243ab2ae-3a84-4a3a-a354-f7e9323130bd -> 243ab2ae-3a84-4a3a-a354-f7e9323130bd
✅ 8119097c-495f-4697-9bcb-aede60a3fc19 : d6bf6972-27d6-4018-ab24-80c5b28a9350 -> d6bf6972-27d6-4018-ab24-80c5b28a9350
✅ 2a429fea-4433-4df2-b1a8-4dee281d752b : 371ee832-3fc4-4c22-a45a-833a07a0efeb -> 371ee832-3fc4-4c22-a45a-833a07a0efeb
✅ 8c302392-e96d-440a-a8ee-220f8be0990e : 4da847ff-c0c5-41d3-aff1-0b0e34e8c8a5 -> 4da847ff-c0c5-41d3-aff1-0b0e34e8c8a5
✅ 2352b7a9-82bd-4c3c-9462-e2685d19dc8d : 3cc17bf2-c209-4c0b-833d-c28842a96763 -> 3cc17bf2-c209-4c0b-833d-c28842a96763
--- n: 100  ---
✅ 196845b6-70e4-447d-adf0-47b1c9c0e9a8 : NOT_FOUND -> NOT_FOUND
❌ 0876ba2c-ee0e-4e05-bc7e-7464ef3f1df9 : 243ab2ae-3a84-4a3a-a354-f7e9323130bd -> NOT_FOUND
❌ 5a8f1ba6-2382-4011-aff8-b32b32608211 : cc98e7a8-e616-49aa-9375-0b39e42b449a -> cc7a4d73-af46-4e3d-ae31-cbbdbfaad8c0
❌ e

In [ ]:
model = "openai:gpt-4o-mini" #context size 128k
#5000 = 250k tokens
_test(model, batches=[500, 2500])


===== model: openai:gpt-4o-mini =====
--- n: 500  ---
✅ 296845b6-70e4-447d-adf0-47b1c9c0e9a8 : NOT_FOUND -> NOT_FOUND
✅ 0876ba2c-ee0e-4e05-bc7e-7464ef3f1df9 : 243ab2ae-3a84-4a3a-a354-f7e9323130bd -> 243ab2ae-3a84-4a3a-a354-f7e9323130bd
✅ bbdb2192-7f3c-46e3-bd4a-49f24a93c118 : 039955f2-5547-4c3b-a4aa-851dcb3783e3 -> 039955f2-5547-4c3b-a4aa-851dcb3783e3
✅ 82118d34-1e83-4735-81d6-785195538ceb : 364655fc-939c-4197-b99d-4ca0caef6ea8 -> 364655fc-939c-4197-b99d-4ca0caef6ea8
✅ 2fe14d64-8972-4344-a82c-38f861454221 : 1142407a-a57c-472c-89dd-6a3e3f6a878d -> 1142407a-a57c-472c-89dd-6a3e3f6a878d
✅ 11c501df-accd-448e-845c-ade6d5078a4c : b2943eec-e008-4fec-8432-8bcd1cebc645 -> b2943eec-e008-4fec-8432-8bcd1cebc645
--- n: 2500  ---
✅ 296845b6-70e4-447d-adf0-47b1c9c0e9a8 : NOT_FOUND -> NOT_FOUND
✅ 0876ba2c-ee0e-4e05-bc7e-7464ef3f1df9 : 243ab2ae-3a84-4a3a-a354-f7e9323130bd -> 243ab2ae-3a84-4a3a-a354-f7e9323130bd
❌ 64e15536-accd-45f3-9bbd-055bc5b33a0d : 654a8b8b-7c5b-4e90-98f5-056968a75f95 -> cfd512b9-f4

BadRequestError: Error code: 400 - {'error': {'message': "This model's maximum context length is 128000 tokens. However, your messages resulted in 252746 tokens. Please reduce the length of the messages.", 'type': 'invalid_request_error', 'param': 'messages', 'code': 'context_length_exceeded'}}

In [ ]:
model = "openai:gpt-4.1-nano" #context size 1M
_test(model, batches=[500, 2500, 5000])


===== model: openai:gpt-4.1-nano =====
--- n: 500  ---
❌ 296845b6-70e4-447d-adf0-47b1c9c0e9a8 : NOT_FOUND -> e13dcb96-8652-4a88-bbb6-408c9f29bd35
✅ 0876ba2c-ee0e-4e05-bc7e-7464ef3f1df9 : 243ab2ae-3a84-4a3a-a354-f7e9323130bd -> 243ab2ae-3a84-4a3a-a354-f7e9323130bd
❌ bbdb2192-7f3c-46e3-bd4a-49f24a93c118 : 039955f2-5547-4c3b-a4aa-851dcb3783e3 -> 54d61910-1830-4ca1-b922-4bc78a8530bf
✅ 82118d34-1e83-4735-81d6-785195538ceb : 364655fc-939c-4197-b99d-4ca0caef6ea8 -> 364655fc-939c-4197-b99d-4ca0caef6ea8
✅ 2fe14d64-8972-4344-a82c-38f861454221 : 1142407a-a57c-472c-89dd-6a3e3f6a878d -> 1142407a-a57c-472c-89dd-6a3e3f6a878d
✅ 11c501df-accd-448e-845c-ade6d5078a4c : b2943eec-e008-4fec-8432-8bcd1cebc645 -> b2943eec-e008-4fec-8432-8bcd1cebc645
--- n: 2500  ---
✅ 296845b6-70e4-447d-adf0-47b1c9c0e9a8 : NOT_FOUND -> NOT_FOUND
✅ 0876ba2c-ee0e-4e05-bc7e-7464ef3f1df9 : 243ab2ae-3a84-4a3a-a354-f7e9323130bd -> 243ab2ae-3a84-4a3a-a354-f7e9323130bd
✅ 64e15536-accd-45f3-9bbd-055bc5b33a0d : 654a8b8b-7c5b-4e90-98f5

In [9]:
# ✅
model = "openai:gpt-4.1" #context size 1M
_test(model, batches=[1000, 5000])


===== model: openai:gpt-4.1 =====
--- n: 1000  ---
✅ 196845b6-70e4-447d-adf0-47b1c9c0e9a8 : NOT_FOUND -> NOT_FOUND
✅ 0876ba2c-ee0e-4e05-bc7e-7464ef3f1df9 : 243ab2ae-3a84-4a3a-a354-f7e9323130bd -> 243ab2ae-3a84-4a3a-a354-f7e9323130bd
✅ f5b97faa-197a-4cf4-8d49-a772bbc7894d : f660941c-8dd2-4c18-93ef-967135ed909a -> f660941c-8dd2-4c18-93ef-967135ed909a
✅ 4095f6b5-4836-4e8e-b196-2c60b4d59c53 : 85f95ae4-ee0b-4b00-af1d-d0281d21cd24 -> 85f95ae4-ee0b-4b00-af1d-d0281d21cd24
✅ bafa1203-54a3-4168-9461-5a415bd71936 : 523adede-e0cf-48ab-8525-c9a0ddad38fb -> 523adede-e0cf-48ab-8525-c9a0ddad38fb
✅ 11307446-23bb-4882-8c40-959a10d9aa33 : ca90c076-3fe1-4207-97bf-f08fe2257f4e -> ca90c076-3fe1-4207-97bf-f08fe2257f4e
--- n: 5000  ---
✅ 296845b6-70e4-447d-adf0-47b1c9c0e9a8 : NOT_FOUND -> NOT_FOUND
✅ 0876ba2c-ee0e-4e05-bc7e-7464ef3f1df9 : 243ab2ae-3a84-4a3a-a354-f7e9323130bd -> 243ab2ae-3a84-4a3a-a354-f7e9323130bd
✅ 088b098d-cf48-4118-ac9e-b1bb77353dde : e8a09be5-2a11-4402-9c99-656015c36168 -> e8a09be5-2a11-

In [3]:
from transformers import pipeline
import torch

model_id = "ibm-granite/granite-4.0-micro" 
pipe = pipeline(
    "text-generation",
    model=model_id,
    torch_dtype="auto",
    device_map="auto",
    do_sample=False,
    max_new_tokens=1024,
    batch_size=4,  
)

for num in [ 500, 1000]:
    _data = {k: data[k] for k in list(data.keys())[:num]}
    token_count = len(json.dumps(_data))
    
    first_needle = dict(list(_data.items())[:1])
    middle_needle = dict(list(_data.items())[num // 2:num // 2 + 1])
    third_quarter_needle = dict(list(_data.items())[3 * num // 4:3 * num // 4 + 1])
    before_final_needle = dict(list(_data.items())[-2:-1])
    final_needle = dict(list(_data.items())[-1:])
    not_found_needle = {"196845b6-70e4-447d-adf0-47b1c9c0e9a8": "NOT_FOUND"}
    
    needle_dicts = [not_found_needle, first_needle, middle_needle, 
                    third_quarter_needle, before_final_needle, final_needle]
    
    # Prepare all prompts at once
    all_messages = []
    for needle_dict in needle_dicts:
        needle = list(needle_dict.keys())[0]
        prompt = prompt_template.format(data=json.dumps(_data, indent=2), key=needle)
        messages = [{"role": "user", "content": prompt}]
        all_messages.append(messages)
    
    # Process all prompts in batch
    print(f"--- n: {num} ---")
    outputs = pipe(all_messages, temperature=0.01, do_sample=False, batch_size=4)
    
    # Process results
    for needle_dict, output in zip(needle_dicts, outputs):
        needle = list(needle_dict.keys())[0]
        response = output[0]["generated_text"][-1]["content"].strip()        
        print(f"{'✅' if (needle_dict[needle] == response) else '❌'} {needle} : {needle_dict[needle]} -> {response}")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


--- n: 500 ---
✅ 196845b6-70e4-447d-adf0-47b1c9c0e9a8 : NOT_FOUND -> NOT_FOUND
✅ 0876ba2c-ee0e-4e05-bc7e-7464ef3f1df9 : 243ab2ae-3a84-4a3a-a354-f7e9323130bd -> 243ab2ae-3a84-4a3a-a354-f7e9323130bd
✅ bbdb2192-7f3c-46e3-bd4a-49f24a93c118 : 039955f2-5547-4c3b-a4aa-851dcb3783e3 -> 039955f2-5547-4c3b-a4aa-851dcb3783e3
✅ 82118d34-1e83-4735-81d6-785195538ceb : 364655fc-939c-4197-b99d-4ca0caef6ea8 -> 364655fc-939c-4197-b99d-4ca0caef6ea8
✅ 2fe14d64-8972-4344-a82c-38f861454221 : 1142407a-a57c-472c-89dd-6a3e3f6a878d -> 1142407a-a57c-472c-89dd-6a3e3f6a878d
✅ 11c501df-accd-448e-845c-ade6d5078a4c : b2943eec-e008-4fec-8432-8bcd1cebc645 -> b2943eec-e008-4fec-8432-8bcd1cebc645
--- n: 1000 ---
❌ 196845b6-70e4-447d-adf0-47b1c9c0e9a8 : NOT_FOUND -> a4b387a8-057e-47ba-811a-a8176097d5f1
✅ 0876ba2c-ee0e-4e05-bc7e-7464ef3f1df9 : 243ab2ae-3a84-4a3a-a354-f7e9323130bd -> 243ab2ae-3a84-4a3a-a354-f7e9323130bd
❌ f5b97faa-197a-4cf4-8d49-a772bbc7894d : f660941c-8dd2-4c18-93ef-967135ed909a -> f660941c-08e7-4329-b58b-9

In [4]:
if "pipe" in locals():
    del pipe
    torch.cuda.empty_cache()